# Lezione 2 — Embeddings & ricerca semantica

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ccasadei-maggioli/corso-nlp-genai-2026/blob/main/lezione_2_embeddings/notebook_02_embeddings.ipynb)

Bentornati! 👋 Nella Lezione 1 abbiamo classificato il **sentiment** delle recensioni
con una pipeline in una riga. Oggi facciamo un passo più in profondità: vediamo come
trasformare il **testo in numeri** (gli **embeddings**) e come usarli per la
**ricerca semantica** — cioè cercare *per significato*, non per parola esatta.

In questa lezione:
1. capiamo cosa sono gli **embeddings** e lo **spazio vettoriale (latente)**;
2. misuriamo quanto due testi si somigliano con la **similarità coseno**;
3. calcoliamo gli embeddings di tutte le recensioni con `intfloat/multilingual-e5-base`;
4. costruiamo un piccolo **motore di ricerca semantica** sulle recensioni;
5. (bonus) visualizziamo gli embeddings in **2D** con PCA.

> 🎯 **Filo conduttore:** continuiamo a lavorare sulle **recensioni clienti in italiano**.
> Gli embeddings di oggi sono il mattone su cui costruiremo il **retrieval** del progetto
> finale (Lezione 6, RAG).

---
### ⚙️ Prima di tutto: attiva la GPU T4
Menu **`Runtime` → `Change runtime type` → Hardware accelerator: `T4 GPU` → `Save`**.
Il calcolo degli embeddings è molto più veloce su GPU.

## 1. Installiamo le librerie

Useremo **`sentence-transformers`**, la libreria che rende immediato calcolare gli
embeddings con i modelli di Hugging Face. Aggiungiamo `scikit-learn` (similarità,
PCA) e `matplotlib` (grafici). `torch` e `pandas` sono già presenti su Colab.

> 💡 Ogni notebook del corso installa da solo ciò che gli serve, così puoi aprire
> qualsiasi lezione in modo indipendente.

In [ ]:
# -q = silenzioso. Su Colab l'installazione richiede qualche secondo.
!pip install -q "sentence-transformers>=3.0" "scikit-learn>=1.3" matplotlib "sentencepiece>=0.2" "protobuf>=4.0"
print("Librerie installate ✅")

## 2. Cosa sono gli embeddings? 🧠

Un **embedding** è la rappresentazione di un testo come **vettore di numeri**
(una lista di float, qui di lunghezza 768). L'idea chiave:

> Testi con **significato simile** finiscono **vicini** nello spazio dei vettori;
> testi con significato diverso finiscono **lontani**.

Questo spazio numerico in cui "vivono" i vettori si chiama **spazio vettoriale** (o
**spazio latente**): non lo possiamo vedere direttamente (ha centinaia di dimensioni),
ma possiamo *misurare le distanze* tra i punti. È proprio ciò che ci permette di
cercare **per significato** invece che per parola esatta.

- 🔗 **Esplora gli embeddings in 3D** con il TensorFlow Embedding Projector:
  <https://projector.tensorflow.org> — ruota la nuvola di punti e nota come le parole
  con significato affine si raggruppano insieme.

**A cosa servono?** Ricerca semantica, raggruppamento (clustering), sistemi di
raccomandazione e — soprattutto per noi — il **retrieval** alla base del **RAG**:
dato un testo, trovare i documenti più pertinenti del nostro archivio.

## 3. La similarità coseno 📐

Per misurare *quanto due vettori sono vicini* usiamo la **similarità coseno**:
guarda l'**angolo** tra i due vettori, ignorando la loro lunghezza.

- vale **1** quando i due testi hanno significato (quasi) identico → stessa direzione;
- vale **~0** quando non c'entrano niente l'uno con l'altro → vettori "ortogonali";
- più è alta, più i testi sono simili.

> 🔧 **Trucco pratico:** se *normalizziamo* gli embeddings (lunghezza = 1), la
> similarità coseno diventa un semplice **prodotto scalare**. Per questo più avanti
> useremo `normalize_embeddings=True`: ci semplifica i calcoli.

## 4. Il nostro dataset: le recensioni 🛒

Riprendiamo lo **stesso** dataset sintetico di recensioni in italiano della Lezione 1,
generato dallo script del repository (`dati/genera_recensioni.py`). È *riproducibile*
(seed fisso): otteniamo sempre le stesse 200 recensioni.

Ricorda lo schema delle colonne: `id, data, prodotto, categoria, rating, titolo, testo`.

In [ ]:
import os, torch
import pandas as pd

# Scarica lo script generatore se non è già nella sessione Colab.
if not os.path.exists("genera_recensioni.py"):
    !wget -q https://raw.githubusercontent.com/ccasadei-maggioli/corso-nlp-genai-2026/main/dati/genera_recensioni.py

import genera_recensioni

df = pd.DataFrame(genera_recensioni.genera_recensioni(n=200, seed=42))

print("Numero di recensioni:", len(df))
df.head(3)

## 5. Carichiamo il modello di embeddings 🤖

Useremo **`intfloat/multilingual-e5-base`**: un modello *multilingue* (capisce bene
l'italiano) e di dimensioni contenute, perfetto per la GPU T4. Produce vettori di
**768 dimensioni**.

Lo carichiamo con `SentenceTransformer`, indicando di usare la **GPU** se disponibile.

In [ ]:
from sentence_transformers import SentenceTransformer

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", dispositivo)

modello = SentenceTransformer("intfloat/multilingual-e5-base", device=dispositivo)
print("Modello caricato ✅ — dimensione embedding:", modello.get_sentence_embedding_dimension())

## 6. ⚠️ Dettaglio importante: i prefissi di e5

La famiglia di modelli **e5** è stata addestrata con dei **prefissi** che vanno
anteposti al testo, per distinguere i due ruoli:

- per i **documenti** (le recensioni da archiviare/indicizzare) → `"passage: "`
- per le **query** (la domanda di ricerca dell'utente) → `"query: "`

Se dimentichi i prefissi, il modello funziona comunque, ma la qualità della ricerca
**peggiora sensibilmente**. È un dettaglio specifico di questa famiglia di modelli:
li mettiamo sempre.

## 7. Calcoliamo gli embeddings di tutte le recensioni

Passiamo al modello i testi delle recensioni — ciascuno con il prefisso
`"passage: "` — e otteniamo una **matrice**: una riga per recensione, una colonna
per dimensione. Con 200 recensioni e vettori da 768 valori, la forma (`shape`) sarà
**(200, 768)**.

Usiamo `normalize_embeddings=True` (vedi il trucco della similarità coseno).

In [ ]:
# Anteponiamo il prefisso "passage: " a ogni recensione (sono i nostri documenti).
documenti = ["passage: " + testo for testo in df["testo"]]

embeddings = modello.encode(
    documenti,
    normalize_embeddings=True,   # vettori di lunghezza 1 -> coseno = prodotto scalare
    show_progress_bar=True,
)

print("Tipo:", type(embeddings))
print("Forma della matrice (n_recensioni, n_dimensioni):", embeddings.shape)

## 8. Similarità coseno tra recensioni di esempio

Prendiamo tre recensioni e misuriamo quanto si somigliano *a coppie*. Usiamo l'utility
`util.cos_sim` di `sentence-transformers`. Poiché gli embeddings sono già normalizzati,
i valori sono direttamente le similarità coseno.

In [ ]:
from sentence_transformers import util

# Scegliamo tre recensioni (indici a piacere) e mostriamone il testo.
indici = [0, 1, 2]
for i in indici:
    print(f"[{i}] ({df.iloc[i]['rating']}★) {df.iloc[i]['testo']}")

# Matrice di similarità coseno fra le tre recensioni.
sotto_insieme = embeddings[indici]
matrice_sim = util.cos_sim(sotto_insieme, sotto_insieme)

print("\nMatrice di similarità coseno (3x3):")
print(matrice_sim.round(decimals=3))

## 9. Ricerca semantica: il cuore della lezione 🔎

Ora il pezzo più interessante. Vogliamo, **data una domanda in italiano**, trovare le
recensioni più pertinenti — anche se non contengono le **stesse parole** della domanda.

Il procedimento:
1. trasformiamo la query in embedding (con il prefisso `"query: "`);
2. calcoliamo la similarità coseno tra la query e **tutte** le recensioni;
3. prendiamo le **top-k** con punteggio più alto.

`util.semantic_search` fa esattamente questo in modo efficiente. Lo incapsuliamo in
una funzione `cerca(query, k=5)` riutilizzabile.

In [ ]:
from sentence_transformers import util

def cerca(query, k=5):
    """Restituisce le k recensioni più simili (per significato) alla query.

    Ritorna un DataFrame con rating, prodotto, punteggio di similarità e testo.
    """
    # 1) Embedding della query, con il prefisso "query: " richiesto da e5.
    emb_query = modello.encode(
        "query: " + query,
        normalize_embeddings=True,
    )
    # 2) Ricerca dei top-k documenti più simili nell'archivio degli embeddings.
    risultati = util.semantic_search(emb_query, embeddings, top_k=k)[0]

    # 3) Costruiamo un DataFrame leggibile con i risultati.
    righe = []
    for r in risultati:
        i = r["corpus_id"]
        righe.append({
            "punteggio": round(float(r["score"]), 3),
            "rating": df.iloc[i]["rating"],
            "prodotto": df.iloc[i]["prodotto"],
            "testo": df.iloc[i]["testo"],
        })
    return pd.DataFrame(righe)


# Proviamo con una domanda sulla logistica.
query = "problemi con la spedizione e i tempi di consegna"
risultati = cerca(query, k=5)
print(f"Query: {query}\n")
risultati

## 10. Semantica vs parola chiave 🆚

La ricerca semantica trova recensioni pertinenti **anche senza le parole esatte** della
query. Confrontiamola con una banale ricerca per parola chiave (`str.contains`).

Cerchiamo recensioni che parlano di **costo elevato** con la query
`"il prezzo è troppo alto"`. Una ricerca testuale per la parola *"caro"* perderebbe
le recensioni che dicono *"troppo caro"*, *"non è giustificato dalla qualità"* o
*"si trova di meglio allo stesso prezzo"* — ma la ricerca semantica le coglie.

In [ ]:
# A) Ricerca SEMANTICA (per significato).
print("=== Ricerca semantica: 'il prezzo è troppo alto' ===")
semantica = cerca("il prezzo è troppo alto", k=5)
display(semantica[["punteggio", "rating", "testo"]])

# B) Ricerca per PAROLA CHIAVE (str.contains): trova solo la parola 'caro'.
print("\n=== Ricerca per parola chiave: testo che contiene 'caro' ===")
parola_chiave = df[df["testo"].str.contains("caro", case=False)]
print(f"Recensioni trovate con 'caro': {len(parola_chiave)}")
display(parola_chiave[["rating", "testo"]].head(5))

print(
    "\n👉 Nota: la ricerca per parola chiave trova SOLO chi scrive esattamente 'caro', "
    "mentre quella semantica recupera anche 'prezzo non giustificato' o "
    "'si trova di meglio allo stesso prezzo'."
)

## 11. (Bonus) Visualizziamo gli embeddings in 2D 🎨

I nostri vettori hanno 768 dimensioni: impossibili da disegnare. Con la **PCA**
(Principal Component Analysis) li *proiettiamo* in 2 dimensioni mantenendo il più
possibile la struttura. Coloriamo ogni punto per **rating**: spesso le recensioni
molto positive (5★) e molto negative (1★) si dispongono in zone diverse del piano.

> ℹ️ La PCA è una semplificazione drastica (da 768 a 2 dimensioni): serve a *intuire*
> la struttura, non a misurarla con precisione. Per esplorazioni più fini esiste il
> già citato **Embedding Projector** (<https://projector.tensorflow.org>).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Riduciamo da 768 a 2 dimensioni.
pca = PCA(n_components=2)
punti_2d = pca.fit_transform(embeddings)

plt.figure(figsize=(8, 6))
grafico = plt.scatter(
    punti_2d[:, 0],
    punti_2d[:, 1],
    c=df["rating"],
    cmap="RdYlGn",   # rosso = poche stelle, verde = molte stelle
    alpha=0.8,
)
plt.colorbar(grafico, label="rating (stelle)")
plt.title("Recensioni proiettate in 2D con PCA (colore = rating)")
plt.xlabel("Componente principale 1")
plt.ylabel("Componente principale 2")
plt.show()

## 12. Esercizio 🏋️

Mettiamo in pratica due cose:

**A)** Cambia la **query** e il valore di **k** nella funzione `cerca(...)` e osserva
come cambiano i risultati (prova ad esempio `"assistenza clienti che non risponde"`).

**B)** Implementa la **similarità recensione-recensione**: data una recensione di
riferimento (per indice), trova le **altre** recensioni più simili a *quella*.
Attenzione: la recensione di riferimento è già un **documento**, quindi è già nei nostri
`embeddings` — non serve ricalcolarla, ed escludiamola dai risultati (avrebbe
similarità 1 con sé stessa).

Completa la funzione `simili_a(indice, k=5)` dove indicato dal `TODO`.

In [ ]:
from sentence_transformers import util

def simili_a(indice, k=5):
    """Trova le k recensioni più simili a quella di riferimento (per indice)."""
    # L'embedding di riferimento è già pronto nella matrice 'embeddings'.
    emb_riferimento = embeddings[indice]

    # TODO: calcola la similarità tra l'embedding di riferimento e TUTTI gli
    #       embeddings, poi prendi i top-(k+1) risultati (uno sarà la recensione
    #       stessa, che escluderemo). Suggerimento: util.semantic_search(...).
    # --- SOLUZIONE ---
    risultati = util.semantic_search(emb_riferimento, embeddings, top_k=k + 1)[0]

    righe = []
    for r in risultati:
        i = r["corpus_id"]
        if i == indice:
            continue  # saltiamo la recensione di riferimento (similarità 1 con sé stessa)
        righe.append({
            "punteggio": round(float(r["score"]), 3),
            "rating": df.iloc[i]["rating"],
            "prodotto": df.iloc[i]["prodotto"],
            "testo": df.iloc[i]["testo"],
        })
    return pd.DataFrame(righe[:k])


# Recensione di riferimento.
riferimento = 0
print("Recensione di riferimento:")
print(f"[{riferimento}] ({df.iloc[riferimento]['rating']}★) {df.iloc[riferimento]['testo']}\n")

print("Recensioni più simili:")
simili_a(riferimento, k=5)

## 13. Riepilogo e prossimi passi ✅

Oggi abbiamo:
- capito cosa sono gli **embeddings** e lo **spazio vettoriale (latente)**;
- misurato la somiglianza tra testi con la **similarità coseno**;
- trasformato tutte le recensioni in una **matrice (200, 768)** con
  `intfloat/multilingual-e5-base` (ricordando i prefissi `passage:` / `query:`);
- costruito una funzione `cerca(query, k)` per la **ricerca semantica**, vista
  vincere contro la ricerca per parola chiave;
- (bonus) visualizzato gli embeddings in **2D** con PCA.

🔗 **Aggancio al progetto finale (Lezione 6):** ciò che oggi chiamiamo `cerca(...)` è
esattamente il **retrieval** di un sistema **RAG**: dato il messaggio dell'utente,
recuperare i documenti più pertinenti da dare in pasto a un LLM. Gli embeddings di oggi
sono quindi il cuore del motore di ricerca che useremo nell'app finale.

➡️ **Prossima lezione (Lezione 3):** i modelli **task-specific** — analisi del
**sentiment** più avanzata, **NER** (estrazione di entità) e **classificazione** dei
temi delle recensioni.

📦 Tutto il materiale del corso: https://github.com/ccasadei-maggioli/corso-nlp-genai-2026